# MINT layer sweep for the contact head

Which MINT layer works best for inter-chain contact prediction? This notebook trains the same
`contact/train_contact_head.py` head once per layer and compares them on PPLM's validation set.

1. **Train pairs.** By default it reuses embeddings you already saved (`REUSE_TRAIN_EMB`), taking
   the pairs in them. With `REUSE_TRAIN_EMB = None` it samples `N_TRAIN` clusters (one pair each) from PPLM
   and extracts them. Either way, `N_VAL` validation pairs are sampled from PPLM.
2. Label the pairs with `contact/label_contacts.py` (downloads the structures from RCSB).
3. Extract embeddings for whatever isn't saved yet. With reused train embeddings, that's only the validation pairs.
4. Train one head per layer (`--layers L`), plus optionally one on all of them with the learned layer mix.
5. Compare best validation AP / top-L/5 precision.

**Runtime → Change runtime type → GPU** (T4 is enough).

The branch in `BRANCH` must contain `contact/train_contact_head.py`, `contact/label_contacts.py` and this
notebook's other dependencies, so push them first.

Re-running is safe after a disconnect. Pair lists, contact labels, embeddings and finished runs are
saved on Drive and reused, so each step only ever runs once. Each session copies the embeddings to the
Colab disk before training, because reading them straight from Drive is slow.

In [ ]:
# ---- settings ----
DRIVE_DIR  = "/content/drive/MyDrive/mint"
CKPT       = f"{DRIVE_DIR}/mint.ckpt"               # cached so it's only downloaded once (~3.2 GB)
PPLM_TAR   = f"{DRIVE_DIR}/PPLM_Dataset.tar.gz"     # cached copy of the PPLM download
SWEEP_DIR  = f"{DRIVE_DIR}/layer_sweep"             # pair lists, contact labels, runs, results
EMB_DIR    = f"{SWEEP_DIR}/emb"                     # embeddings extracted here are saved and reused
LOCAL_EMB  = "/content/emb"                         # training reads a local copy (much faster than Drive); None = read Drive directly

REPO_URL   = "https://github.com/AlecSilver/mint.git"
BRANCH     = "interaction_head"

# data
REUSE_TRAIN_EMB = f"{DRIVE_DIR}/emb_train_multilayer_test"  # saved train shards to reuse; None = sample + extract N_TRAIN
SWEEP_LAYERS = [24, 30, 33]   # must all be saved in REUSE_TRAIN_EMB; 33 = final layer
MIX_RUN      = True    # also train one head on all SWEEP_LAYERS with the learned layer mix
N_TRAIN      = 2000    # only without REUSE_TRAIN_EMB: train clusters (one random pair each) from trainset.PDB.txt
N_VAL        = 500     # pairs from validset.PDB.txt
DATA_SEED    = 0       # which pairs are sampled
CUTOFF       = 8.0     # Å, contact definition
MAX_TOKENS   = 16384   # extraction token budget per forward pass; lower it on CUDA out-of-memory
CROP         = "interface"  # extraction window for chains > 510 residues; with REUSE_TRAIN_EMB, its crop is used instead

# head training: identical for every run, so only the layers differ
PAIR       = "add"     # add | multiply
HIDDEN     = 128
EPOCHS     = 10
BATCH_SIZE = 4
TRAIN_CROP = 256
ALL_PAIRS  = True      # every train pair each epoch; False = one per PPLM cluster (too few with the reused 1000 pairs, ~125 clusters)
TRAIN_SEED = 0         # change it and re-run to see how much of a gap between layers is noise
TRAIN_ARGS = ""        # anything else for train_contact_head.py, e.g. "--dist-weight 0.5"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs(SWEEP_DIR, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Code, checkpoint and PPLM data

If the repo is private, use a URL with a personal access token
(`https://<token>@github.com/AlecSilver/mint.git`).

In [ ]:
REPO_DIR = "/content/mint"
if not os.path.exists(REPO_DIR):
    !git clone -q --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
HEAD_DIR = f"{REPO_DIR}/interaction_head"
for f in ["contact/label_contacts.py", "extract_residue_embeddings.py", "contact/train_contact_head.py"]:
    assert os.path.exists(f"{HEAD_DIR}/{f}"), f"{f} is missing on branch {BRANCH}; push it first"

# Colab already has torch, numpy, scipy, pandas and tqdm; label_contacts.py also needs gemmi
!pip install -q --no-deps -e {REPO_DIR}
!pip install -q gemmi

In [ ]:
if not os.path.exists(CKPT):
    !wget -q --show-progress -O {CKPT}.part https://huggingface.co/varunullanat2012/mint/resolve/main/mint.ckpt
    os.replace(CKPT + ".part", CKPT)

if not os.path.exists(PPLM_TAR):
    !wget -q --show-progress -O {PPLM_TAR}.part https://zhanggroup.org/PPLM/bin/databases/PPLM_Dataset.tar.gz
    os.replace(PPLM_TAR + ".part", PPLM_TAR)
PPLM = "/content/PPLM_Dataset"
if not os.path.exists(PPLM):
    !tar -xzf {PPLM_TAR} -C /content
!ls {PPLM}

## 1. Pairs

With `REUSE_TRAIN_EMB`, the train list is the pairs in those shards, kept in their PPLM clusters.
Contacts have to be labelled on the same sequences the embeddings were made from, so this also checks
whether the interface JSONL behind them was made with `--seq-dir` (PPLM's `.fa` sequences) or without.

File names include the sizes and seed, so changing them makes new files instead of reusing stale ones.

In [ ]:
import glob, json, random, sys
import torch
sys.path.insert(0, f"{HEAD_DIR}/contact")
from label_contacts import PAIR_RE, read_fasta

def write_list(dst, clusters):
    """One line per cluster, its pairs space-separated (the PPLM list format)."""
    with open(dst + ".part", "w") as f:
        f.writelines(" ".join(c) + "\n" for c in clusters)
    os.replace(dst + ".part", dst)

def sample_pairs(src, n, dst, seed):
    """n random clusters from a PPLM list, one random pair each."""
    if os.path.exists(dst):
        print("exists:", dst)
        return
    with open(src) as f:
        clusters = [[m.group(0) for m in PAIR_RE.finditer(line)] for line in f]
    clusters = [c for c in clusters if c]
    rng = random.Random(seed)
    picked = rng.sample(clusters, min(n, len(clusters)))
    write_list(dst, [[rng.choice(c)] for c in picked])
    print(f"{len(picked)} of {len(clusters)} clusters -> {dst}")

EMB = {}  # split -> embedding folder on Drive
if REUSE_TRAIN_EMB:
    with open(f"{REUSE_TRAIN_EMB}/meta.json") as f:
        reuse_meta = json.load(f)
    missing = set(SWEEP_LAYERS) - set(reuse_meta["layers"])
    assert not missing, f"layers {sorted(missing)} aren't in {REUSE_TRAIN_EMB} (saved: {reuse_meta['layers']})"
    CROP = reuse_meta["crop"]  # extract validation the same way
    TAG = f"{os.path.basename(REUSE_TRAIN_EMB)}_v{N_VAL}_s{DATA_SEED}"
    EMB["train"] = REUSE_TRAIN_EMB

    # pair ids and clusters straight from the shards (mmap: the embeddings themselves aren't read)
    by_cluster = {}
    for path in sorted(glob.glob(f"{REUSE_TRAIN_EMB}/shard_*.pt")):
        shard = torch.load(path, map_location="cpu", mmap=True, weights_only=False)
        for pid, c in zip(shard["ids"], shard["cluster"].tolist()):
            by_cluster.setdefault(c, []).append(pid)
    train_ids = {p for c in by_cluster.values() for p in c}
    print(f"{REUSE_TRAIN_EMB}: {len(train_ids)} pairs in {len(by_cluster)} clusters, layers {reuse_meta['layers']}, crop {CROP}")
    if not os.path.exists(f"{SWEEP_DIR}/train_{TAG}.txt"):
        write_list(f"{SWEEP_DIR}/train_{TAG}.txt", by_cluster.values())

    # which sequences were the embeddings made from?
    seqs = {}
    with open(reuse_meta["input"]) as f:
        for line in f:
            rec = json.loads(line)
            if rec["id"] in train_ids:
                seqs[rec["id"]] = rec["seq1"]
                if len(seqs) == len(train_ids):
                    break
    check = list(seqs.items())[:200]
    def fa(pid):
        prefix, c1, _ = pid.rsplit("_", 2)
        path = f"{PPLM}/PDB_seqs/{prefix}_{c1}.fa"
        return read_fasta(path) if os.path.exists(path) else None
    frac = sum(fa(pid) == s for pid, s in check) / max(len(check), 1)
    USE_SEQ_DIR = frac > 0.5
    print(f"{frac:.0%} of checked sequences match PPLM's .fa files -> label "
          + ("with --seq-dir" if USE_SEQ_DIR else "without --seq-dir (deposited sequences)"))
else:
    TAG = f"n{N_TRAIN}_v{N_VAL}_s{DATA_SEED}"
    USE_SEQ_DIR = True
    sample_pairs(f"{PPLM}/trainset.PDB.txt", N_TRAIN, f"{SWEEP_DIR}/train_{TAG}.txt", DATA_SEED)
sample_pairs(f"{PPLM}/validset.PDB.txt", N_VAL, f"{SWEEP_DIR}/val_{TAG}.txt", DATA_SEED)
SPLITS = ["train", "val"]

## 2. Contact labels

Downloads each structure from RCSB, roughly 10-30 minutes for 1000-2500 pairs.
It only runs once per pair list. Pairs that fail (missing chains, no contacts) are listed in `*.failed.txt` and dropped.

In [ ]:
%cd {HEAD_DIR}
SEQ_DIR_ARG = f"--seq-dir {PPLM}/PDB_seqs" if USE_SEQ_DIR else ""
for split in SPLITS:
    out = f"{SWEEP_DIR}/{split}_{TAG}_contacts.json"
    if os.path.exists(out):
        print("exists:", out)
        continue
    !python contact/label_contacts.py {SWEEP_DIR}/{split}_{TAG}.txt {out}.part {SEQ_DIR_ARG} --cutoff {CUTOFF} --cache /content/cif_cache
    assert _exit_code == 0, f"label_contacts.py failed for {split}"
    os.replace(out + ".part", out)

if REUSE_TRAIN_EMB:  # labels must be on the same sequences as the saved embeddings, or training skips the pair
    with open(f"{SWEEP_DIR}/train_{TAG}_contacts.json") as f:
        lens = {r["id"]: [len(r["seq1"]), len(r["seq2"])] for r in map(json.loads, f)}
    ok = n = 0
    for path in sorted(glob.glob(f"{REUSE_TRAIN_EMB}/shard_*.pt")):
        shard = torch.load(path, map_location="cpu", mmap=True, weights_only=False)
        for pid, fl in zip(shard["ids"], shard["full_len"].tolist()):
            n += 1
            ok += lens.get(pid) == fl
    print(f"{ok} of {n} saved train pairs have matching contact labels")
    assert ok > 0.8 * n, "most saved pairs don't match their contact labels; check USE_SEQ_DIR"

## 3. Embeddings

Extracts whatever isn't saved yet (with reused train embeddings, just the validation pairs) to Drive.
A split is skipped once it's complete, and resumes from the last finished shard if it was cut off.
The folder name includes the layers and crop mode, so changing either extracts a new set.

In [ ]:
LAYERS_STR = " ".join(map(str, sorted(SWEEP_LAYERS)))
EMB_NAME = f"{TAG}_L{'-'.join(map(str, sorted(SWEEP_LAYERS)))}_{CROP}"
for split in SPLITS:
    if split in EMB:
        print(f"{split}: using saved {EMB[split]}")
        continue
    EMB[split] = out = f"{EMB_DIR}/{split}_{EMB_NAME}"
    if os.path.exists(f"{out}/DONE"):  # skip loading the 3.2 GB checkpoint just to find nothing to do
        print("exists:", out)
        continue
    !python extract_residue_embeddings.py {SWEEP_DIR}/{split}_{TAG}_contacts.json {out} --checkpoint {CKPT} --max-tokens {MAX_TOKENS} --crop {CROP} --layers {LAYERS_STR}
    assert _exit_code == 0, f"extraction failed for {split}"
    open(f"{out}/DONE", "w").close()

# local copy for training; rsync only copies what's missing, so re-running is cheap
EMB_READ = dict(EMB)
if LOCAL_EMB:
    for split in SPLITS:
        !mkdir -p {LOCAL_EMB} && rsync -a --info=progress2 {EMB[split]} {LOCAL_EMB}/
        assert _exit_code == 0, f"copying {split} embeddings failed (disk full?)"
        EMB_READ[split] = f"{LOCAL_EMB}/{os.path.basename(EMB[split])}"
TRAIN_EMB, VAL_EMB = EMB_READ["train"], EMB_READ["val"]
!du -sh {TRAIN_EMB} {VAL_EMB}

## 4. Train one head per layer

Each run writes `best.pt`, `last.pt` and `log.jsonl` to Drive. A finished run is skipped when you
re-run this cell. An unfinished one (cut off by a disconnect) is restarted from scratch.
The run folder name includes a hash of the training settings, so changing them starts a new sweep.

In [ ]:
import hashlib, shutil

settings = dict(pair=PAIR, hidden=HIDDEN, epochs=EPOCHS, batch=BATCH_SIZE, crop=TRAIN_CROP,
                all_pairs=ALL_PAIRS, seed=TRAIN_SEED, cutoff=CUTOFF, extra=TRAIN_ARGS)
RUN_ROOT = f"{SWEEP_DIR}/runs/{TAG}_{PAIR}_h{HIDDEN}_e{EPOCHS}_" + \
           hashlib.md5(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:6]
os.makedirs(RUN_ROOT, exist_ok=True)
with open(f"{RUN_ROOT}/settings.json", "w") as f:
    json.dump(settings, f, indent=2)

RUNS = {f"L{l}": [l] for l in sorted(SWEEP_LAYERS)}
if MIX_RUN and len(SWEEP_LAYERS) > 1:
    RUNS["mix"] = sorted(SWEEP_LAYERS)
ALL_PAIRS_ARG = "--all-pairs" if ALL_PAIRS else ""

def finished(out):
    log = f"{out}/log.jsonl"
    return os.path.exists(log) and sum(1 for _ in open(log)) >= EPOCHS

for name, layers in RUNS.items():
    out = f"{RUN_ROOT}/{name}"
    if finished(out):
        print(f"{name}: done, skipping")
        continue
    shutil.rmtree(out, ignore_errors=True)  # log.jsonl appends, so never continue a partial run
    layers_str = " ".join(map(str, layers))
    print(f"==== {name}: layers {layers_str}")
    !python contact/train_contact_head.py --train-emb {TRAIN_EMB} --train-contacts {SWEEP_DIR}/train_{TAG}_contacts.json --val-emb {VAL_EMB} --val-contacts {SWEEP_DIR}/val_{TAG}_contacts.json --out {out} --layers {layers_str} --pair {PAIR} --hidden {HIDDEN} --epochs {EPOCHS} --batch-size {BATCH_SIZE} --crop {TRAIN_CROP} --cutoff {CUTOFF} --seed {TRAIN_SEED} {ALL_PAIRS_ARG} {TRAIN_ARGS}
    assert _exit_code == 0, f"training failed for {name}"

## 5. Compare

Each run is scored at its best epoch by validation AP. AP is computed over all validation cells.
Top-L/5 is the share of each pair's L/5 highest-scoring cells that are true contacts (L = shorter chain).

In [ ]:
import pandas as pd

rows = []
for name, layers in RUNS.items():
    log = f"{RUN_ROOT}/{name}/log.jsonl"
    if not os.path.exists(log):
        continue
    epochs = [json.loads(l) for l in open(log)]
    best = max(epochs, key=lambda r: r["val_ap"])
    row = {"run": name, "layers": layers, "epochs_done": len(epochs), "best_epoch": best["epoch"],
           "val_ap": best["val_ap"], "top10": best["val_top10"], "topL/10": best["val_topL/10"],
           "topL/5": best["val_topL/5"], "last_ap": epochs[-1]["val_ap"]}
    if len(layers) > 1:
        row["layer_weights"] = {l: round(w, 3) for l, w in zip(layers, best["layer_weights"])}
    rows.append(row)

results = pd.DataFrame(rows)
results.to_csv(f"{RUN_ROOT}/results.csv", index=False)
print("positive rate (AP of a random guess):", round(epochs[-1]["val_pos_rate"], 4))
results.round(4)

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED, GRID, SERIES = "#1f2328", "#59636e", "#d8dee4", "#2f6fdf"
single = results[results.layers.map(len) == 1].reset_index(drop=True)  # already in layer order
x = [l[0] for l in single.layers]
mix = results[results.run == "mix"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
for ax, col, title in zip(axes, ["val_ap", "topL/5"], ["Validation AP", "Validation top-L/5 precision"]):
    ax.plot(x, single[col], color=SERIES, lw=2, marker="o", ms=7, mec="white", mew=1.5, zorder=3)
    b = single[col].idxmax()
    ax.annotate(f"{single[col][b]:.3f}", (single.layers[b][0], single[col][b]), textcoords="offset points",
                xytext=(0, 9), ha="center", color=INK, fontsize=9)
    if len(mix):
        y = mix[col].iloc[0]
        ax.axhline(y, color=MUTED, lw=1.2, ls="--", zorder=2)
        ax.annotate(f"layer mix {y:.3f}", (x[-1], y), textcoords="offset points", xytext=(0, -13),
                    ha="right", color=MUTED, fontsize=9)
    ax.margins(y=0.25)
    ax.set_title(title, loc="left", color=INK, fontsize=11, pad=12)
    ax.set_xlabel("MINT layer", color=MUTED)
    ax.set_xticks(x)
    ax.grid(axis="y", color=GRID, lw=0.8)
    ax.tick_params(colors=MUTED, length=0)
    for s in ax.spines.values():
        s.set_visible(False)
fig.tight_layout()
fig.savefig(f"{RUN_ROOT}/layer_sweep.png", dpi=150)
plt.show()
print("saved to", RUN_ROOT)